#CafChem Teaching - LLMs and token sampling.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MauricioCafiero/CafChem_Teaching_Notebooks/blob/main/notebooks/LLM_Sampling_CafChem.ipynb)

## This notebook allows you to:
- Load models and tokenizers from Huggingface
- use greedy or temperature-based sampling for token selection
- use a pipeline for model use

## Requirements:
- GPU; Colab free tier, or will also run on Kaggle free tier (turn on accelerator GPU).
- Huggingface token in secrets; may need repo access.

##Import libraries

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from transformers import pipeline
import pandas as pd
import numpy as np
import random

## Define functions

In [ ]:
def setup_decoder(model_name: str):
  '''
    Setup the decoder model. Pulls model and tokenizer from HuggingFace, sets up device.

      Args:
        model_name: model to pull from HuggingFace.
      Returns:
        tokenizer, model, device
  '''
  device = "cuda" if torch.cuda.is_available() else "cpu"
  tokenizer = AutoTokenizer.from_pretrained(model_name)
  model = AutoModelForCausalLM.from_pretrained(model_name).to(device)

  print(f"model setup complete: {model_name}")
  return tokenizer, model, device

def decoder_inference(model, tokenizer, device, input_txt: str, n_steps: int,
                      TEMP = 0.0):
  '''
    Takes input_txt and performs autoregressive inference. Captures the chosen token and
    probability at each step.

      Args:
        model: model to pull from HuggingFace.
        tokenizer: tokenier associated with the model
        device: cpu or cuda
        input_txt: prompt to decoder
        n_steps: how many tokens to use
        TEMP: temperature for inference
        use_ramp: Boolean, use temperature ramp?
      Returns:
        iterations: a list containing the input at each step, the chosen token
                    and probability
  '''
  input_ids = tokenizer(input_txt, return_tensors="pt")["input_ids"].to(device)

  iterations = []

  with torch.no_grad():
      for c in range(n_steps):
          iteration = dict()

          iteration["Input"] = tokenizer.decode(input_ids[0])
          output = model(input_ids=input_ids)

          # Select logits of the first batch and the last token and apply softmax
          next_token_logits = output.logits[0, -1, :]
          next_token_probs = torch.softmax(next_token_logits, dim=-1)

          if TEMP < 0.015:
            token_id = torch.argmax(next_token_probs,axis=-1)
          else:
            scaled_probs = next_token_probs**(1/TEMP)
            scaled_probs = scaled_probs / scaled_probs.sum()

            token_id = np.random.choice(scaled_probs.shape[-1], p=scaled_probs.cpu().numpy())
            token_id = torch.tensor(token_id).to(device)

          token_prob = next_token_probs[token_id].cpu().numpy()
          token_choice = (f"{tokenizer.decode(token_id)} ({100 * token_prob:.2f}%)")

          iteration[f"Choice"] = token_choice

          # Append predicted next token to input
          input_ids = torch.cat([input_ids, token_id.unsqueeze(0).unsqueeze(0)], dim=-1)

          iterations.append(iteration)
  return iterations

## Set up model

In [ ]:
model_name = 'HuggingFaceTB/SmolLM2-360M-Instruct'

tokenizer, model, device = setup_decoder(model_name)

## View output token-by-token

In [ ]:
input_txt = "What country has the best cuisine?"
iterations = decoder_inference(model, tokenizer,device, input_txt, n_steps= 50,
                               TEMP = 0.4)

In [ ]:
iterations

In [ ]:
input_txt = "What country has the best cuisine?"
iterations = decoder_inference(model, tokenizer,device, input_txt, n_steps= 50,
                               TEMP = 0.2)

In [ ]:
iterations

## Use a pipeline

In [ ]:
pipe_llm = pipeline("text-generation", model=model, tokenizer=tokenizer)

In [ ]:
messages = [{"system": "you are a helpful assistant","role": "user", "content": input_txt}]
result = pipe_llm(messages, max_new_tokens=500, temperature=1.0)

In [ ]:
result